In [1]:
import pyEDM
import pandas as pd
import numpy as np
import random

p22 = pd.read_csv('../data/processed_22.csv')
p23 = pd.read_csv('../data/processed_23.csv')
p24 = pd.read_csv('../data/processed_24.csv')
p25 = pd.read_csv('../data/processed_25.csv')
combined = pd.concat([p22, p23, p24, p25], ignore_index=True)

In [8]:
env = pd.read_csv("../data/environment_all.csv")

# could be relevant but too many missing values for ema imputation to be effective
env = env.drop(columns=[
    'fluorescent_dissolved_organic_matter_eco',
    'sea_water_turbidity_eco',
    'waterlevel_predicted_m',
    'mass_concentration_of_oxygen_in_sea_water_seaphox',
    'mole_concentration_of_dissolved_molecular_oxygen_in_sea_water_seaphox',
    'fractional_saturation_of_oxygen_in_sea_water_seaphox',
    'sea_water_ph_reported_on_total_scale_seaphox_external'
])

env = env.set_index("date")

env_features = env.columns.tolist()
env_filled = env.copy()
env_was_imputed = pd.DataFrame(index=env.index)

for col in env_features:
    ema = env[col].ewm(span=30, adjust=False).mean()
    env_filled[col] = env[col].fillna(ema)
    env_was_imputed[col] = env[col].isna()

env_norm = env_filled.copy()

for col in env_features:
    mu = env_filled[col].mean()
    sigma = env_filled[col].std()
    env_norm[col] = (env_filled[col] - mu) / sigma

env_norm.head()

,mass_concentration_of_chlorophyll_in_sea_water_ctd,sea_water_electrical_conductivity_ctd,sea_water_practical_salinity_ctd,sea_water_sigma_t_ctd,sea_water_pressure_ctd,sea_water_temperature_ctd,wind_speed_ms,wind_dir_deg,wind_gust_ms,air_temp_c,baro_mb,waterlevel_verified_m
date,,,,,,,,,,,,
20221001,0.385981,1.505434,0.361481,-0.886685,0.780586,1.526412,-0.299392,0.288541,-0.014636,1.008388,-1.293812,1.006101
20221002,0.717746,1.473695,0.375914,-0.833445,0.518250,1.488042,-0.107666,0.304176,-0.095121,1.223492,-0.784742,0.654136
20221003,1.064841,0.861076,0.326602,-0.349443,0.543234,0.835349,-0.114901,0.714355,-0.069251,1.364369,-0.344867,0.304278
20221004,1.402008,0.352896,0.254438,0.008719,0.330867,0.303306,-0.798605,-0.681725,-0.785000,1.100792,-0.532679,0.347483
20221005,1.677097,0.441099,0.238803,-0.079612,0.180961,0.408132,-0.664758,0.782411,-0.787874,0.808433,-0.370815,0.218921


In [11]:
features = [
    "Lpoly_expected_ml", "Area", "Biovolume", "MajorAxisLength",
    "MinorAxisLength", "Perimeter", "Orientation", "Eccentricity",
    "Solidity", "texture_uniformity", "texture_smoothness",
    "texture_average_gray_level", "texture_entropy",
    "texture_average_contrast", "H90", "H180", "Hflip",
    "Extent", "EquivDiameter", "ConvexArea", "ConvexPerimeter"
]

df = combined[["date"] + features].copy()
df = df.set_index("date")

df_filled = df.copy()
was_imputed = pd.DataFrame(index=df.index)

for col in features:
    ema = df[col].ewm(span=30, adjust=False).mean()
    df_filled[col] = df[col].fillna(ema)
    was_imputed[col] = df[col].isna()

# normalize features
df_norm = df_filled.copy()
for col in features:
    mu = df_filled[col].mean()
    sigma = df_filled[col].std()
    
    df_norm[col] = (df_filled[col] - mu) / sigma

df_mv = df_norm.copy()
df_mv = df_mv.reset_index()
df_mv["t"] = np.arange(1, len(df_mv) + 1)
df_mv = df_mv[["t"] + features]

target = "Lpoly_expected_ml"
predictors = [col for col in features if col != target]

df_mv.head()

,t,Lpoly_expected_ml,Area,Biovolume,MajorAxisLength,MinorAxisLength,Perimeter,Orientation,Eccentricity,Solidity,...,texture_average_gray_level,texture_entropy,texture_average_contrast,H90,H180,Hflip,Extent,EquivDiameter,ConvexArea,ConvexPerimeter
0,1,0.201384,1.072846,0.991371,0.962559,1.142312,1.253251,0.943955,-1.226038,-0.255073,...,-1.162215,0.403222,-1.032855,-0.829121,0.879724,1.142507,0.043578,1.092348,1.130929,1.125286
1,2,0.017849,1.032398,0.944410,0.917268,1.113260,1.046023,0.682442,-1.244344,0.407274,...,-1.123666,0.551147,-0.915583,-0.920090,0.405094,0.727350,0.551032,1.064202,1.024678,1.033443
2,3,-0.144662,0.707059,0.562008,0.665467,0.871493,0.772111,0.758824,-1.075938,0.370572,...,-1.060480,0.881960,-0.805379,-0.897370,0.166330,0.388077,0.426334,0.824788,0.703482,0.799048
3,4,-0.052136,0.431385,0.234261,0.474531,0.663959,0.565346,0.993244,-0.849480,0.263461,...,-1.020033,0.984453,-0.823433,-0.785053,0.183644,0.486358,0.126460,0.625717,0.434294,0.610608
4,5,0.708112,0.786575,0.638292,0.746189,0.942608,0.959883,0.545860,-1.076857,0.005587,...,-1.163194,0.172094,-0.980177,-0.915433,0.380184,0.703193,0.184341,0.892770,0.810891,0.886085


https://sugiharalab.github.io/EDM_Documentation/parameters/

In [89]:
def one_simplex(df, target, features, E=4, Tp=1):
    # Randomly select 3 features (+ the target) for the simplex projection
    chosen_features = random.sample(features, 3)
    columns = [target] + chosen_features
    columns_str = " ".join(columns) # has to be 'space separated' idk ????

    N = len(df)
    res = pyEDM.Simplex(
        dataFrame=df,
        columns=columns_str,
        target=target,
        E=E,
        tau=1,
        Tp=Tp,
        lib=f"1 {N}",
        pred=f"1 {N}"
    )

    obs = res["Observations"].to_numpy()
    pred = res["Predictions"].to_numpy()

    mask = np.isfinite(obs) & np.isfinite(pred)
    obs = obs[mask]
    pred = pred[mask]

    if len(obs) < 10 or np.std(obs) == 0 or np.std(pred) == 0:
        return np.nan, chosen_features

    rho = np.corrcoef(obs, pred)[0, 1]
    rmse = np.sqrt(np.mean((obs - pred) ** 2))
    mae = np.mean(np.abs(obs - pred))
    return rho, rmse, mae, chosen_features

def multiview_big(df, target, features, Tp, n_trials=500):
    results = []

    for i in range(n_trials):
        rho, rmse, mae, chosen = one_simplex(df, target, features, E=4, Tp=Tp)

        results.append({
            "rho": rho,
            "rmse": rmse,
            "mae": mae,
            "features": chosen
        })

    return pd.DataFrame(results)

In [ ]:
from collections import Counter

x = df_mv[target].to_numpy()

summary_rows = []
feature_importance_by_tp = {}

for Tp in range(1, 32):
    mv = multiview_big(df_mv, target, predictors, Tp, n_trials=500)

    acf = abs(pd.Series(x).autocorr(lag=Tp))
    mv["rho_eff"] = mv["rho"] - acf

    # summary stats
    summary_rows.append({
        "Tp": Tp,
        "rho_eff_mean": mv["rho_eff"].mean(),
        "rmse_mean": mv["rmse"].mean(),
        "mae_mean": mv["mae"].mean(),
        "n_models": len(mv)
    })

    # feature importance (top 5%)
    top = mv.nlargest(int(0.05 * len(mv)), "rho_eff")

    counts = Counter()
    for feats in top["features"]:
        for f in feats:
            counts[f] += 1

    importance = pd.Series(counts) / len(top)
    feature_importance_by_tp[Tp] = importance.sort_values(ascending=False)

In [91]:
summary_df = pd.DataFrame(summary_rows)
summary_df

,Tp,rho_eff_mean,rmse_mean,mae_mean,n_models
0,1,0.189587,0.440028,0.083862,500
1,2,0.441783,0.403211,0.078743,500
2,3,0.584699,0.377971,0.080460,500
3,4,0.270783,0.810417,0.181902,500
4,5,0.125533,0.945518,0.216849,500
5,6,0.165825,0.965110,0.226251,500
6,7,0.147915,0.979086,0.236489,500
7,8,0.081031,0.998370,0.240146,500
8,9,0.102518,0.996812,0.239738,500
9,10,0.077253,0.997229,0.243745,500


In [92]:
importance_all = pd.concat(
    feature_importance_by_tp,
    names=["Tp", "Feature"]
).reset_index()

importance_all.columns = ["Tp", "Feature", "Proportion"]
importance_all

,Tp,Feature,Proportion
0,1,Solidity,0.52
1,1,Orientation,0.48
2,1,Area,0.32
3,1,texture_average_gray_level,0.24
4,1,MajorAxisLength,0.24
...,...,...,...
473,31,texture_average_gray_level,0.12
474,31,EquivDiameter,0.12
475,31,Biovolume,0.08
476,31,Eccentricity,0.08
